[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IyadSultan/CCI/blob/main/session6/student/Lab5_Wiki_Approach_Student.ipynb)


# Lab 5: The Wiki Approach — RAG Without Retrieval
## CCI Session 6

**Duration:** 15 minutes

### Clinical Scenario
> Karpathy's idea: instead of vector retrieval, maintain a living wiki of markdown files that the LLM updates incrementally. For Wilms tumor — entity pages (cisplatin.md, stage-iii.md, side-effects.md), an index, and a log. No embeddings. No vector DB. Just markdown.

### Objective
- Understand Karpathy's wiki/catalog approach
- Build a clinical wiki structure for Wilms tumor
- Use an LLM to ingest content into the wiki
- Query the wiki by reading the index then drilling down
- Compare with vector RAG

Reference: https://gist.github.com/karpathy/442a6bf555914893e9891c11519de94f

**Design note:** the wiki trades **embedding search** for **named markdown pages** the LLM maintains. Compare with Labs 1–2: there you *retrieve chunks*; here you *curate files* and navigate via `index.md`.


---
## Setup

Optional: save **`/content/wt.md`** from Lab 1 so ingestion has rich text without re-parsing in this notebook.


In [ ]:
!pip install -q langchain langchain-text-splitters langchain-openai openai langchain-community llama-parse faiss-cpu tiktoken

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
print('OPENAI_API_KEY loaded from Colab secrets.')


---
## Cell 1 — Create wiki structure

Lay out folders and starter files. The wiki trades **vector retrieval** for **named pages** the LLM updates; see solutions for the full pattern and Karpathy’s gist in the intro.


In [ ]:
import os
WIKI_DIR = '/content/wt_wiki'
os.makedirs(WIKI_DIR, exist_ok=True)
os.makedirs(f'{WIKI_DIR}/entities', exist_ok=True)

# TODO: Create initial files:
# - index.md (table of contents)
# - log.md (ingestion history)
# - SCHEMA.md (instructions for LLM maintainer — what entity pages exist, format)
#
# Hints:
# Also create subfolders entities/diseases, entities/drugs, entities/stages,
# entities/procedures, entities/side-effects
# Use open(path, 'w').write('# Title\n...') for each file

## Cell 2 — Wiki maintainer prompt

In [ ]:
WIKI_SCHEMA = """
WIKI STRUCTURE for Wilms Tumor Knowledge:
- entities/diseases/{name}.md  — disease pages (e.g., wilms-tumor.md, neuroblastoma.md)
- entities/drugs/{name}.md     — drug pages (e.g., cisplatin.md, vincristine.md)
- entities/stages/{name}.md    — staging pages (stage-i.md through stage-v.md)
- entities/procedures/{name}.md— procedures
- entities/side-effects/{name}.md — side effects
- index.md — master index with links to all pages
- log.md   — what was ingested when

INGEST RULES:
- For new content, identify entities and update/create their pages
- Cross-link pages with [[wiki links]]
- Update index.md with any new pages
- Append to log.md
"""

## Cell 3 — Ingest a clinical text section into the wiki

In [ ]:
# TODO: Take a chunk from WT.pdf (e.g., "Stage III favorable histology" section)
# Use LLM to:
# 1. Identify which entity pages should be updated (returns list of file paths)
# 2. For each affected page, generate the updated markdown content (preserving cross-references)
# 3. Update index.md with any new pages
# 4. Append to log.md
#
# Hints:
# - Define a Pydantic schema PageEdit { path: str, content: str } and IngestPlan { edits: List[PageEdit], log_entry: str }
# - Use ChatOpenAI(...).with_structured_output(IngestPlan)
# - Pass current index.md + SCHEMA + the new text
# - Apply edits by writing files; append log_entry to log.md

from pydantic import BaseModel
from typing import List

# class PageEdit(BaseModel):
#     path: str
#     content: str
# class IngestPlan(BaseModel):
#     edits: List[PageEdit]
#     log_entry: str

# section1 = '...'   # one chunk from WT.pdf about Stage III favorable histology
# plan = ...         # llm.invoke(...)
# apply edits, update index.md, append log.md

# Show resulting tree:
# for root, dirs, files in os.walk(WIKI_DIR):
#     ...

## Cell 4 — Repeat ingest with another section

In [ ]:
# TODO: Ingest a different section
# Show how existing pages get UPDATED (not duplicated)
#
# Hint: pass the *current contents* of relevant pages back into the prompt
# so the model rewrites them with the new info merged in.

## Cell 5 — Query the wiki

In [ ]:
# TODO:
# 1. LLM reads index.md to identify relevant pages
# 2. LLM reads only those pages (no embeddings, no vector search!)
# 3. LLM generates answer with citations to wiki pages
#
# Test: "What is the treatment regimen and side effects for Stage III favorable histology Wilms tumor?"
#
# Hints:
# step1: ask LLM to return a JSON list of page paths it wants to read, given index.md
# step2: read those files and concatenate
# step3: ask LLM to answer using only those pages, citing them

## Cell 6 — Inspect the wiki

In [ ]:
# Show the file tree
# Show contents of a few entity pages
# Show the log

# Hint:
# for root, dirs, files in os.walk(WIKI_DIR):
#     level = root.replace(WIKI_DIR, '').count(os.sep)
#     print('  '*level + os.path.basename(root) + '/')
#     for f in files: print('  '*(level+1) + f)

## Cell 7 — Compare with vector RAG

In [ ]:
# Same question on a vector RAG built from raw chunks of WT.pdf
# Pros of wiki: deterministic, debuggable, no embedding drift, transparent
# Cons: scales worse, slower per query, requires careful prompts
#
# Hint: build a quick FAISS index over the raw text and compare answers.

## Stretch — Wiki linter

In [ ]:
# Stretch: Add a "lint" step that checks for broken cross-references
#
# Hint: parse [[wiki links]] from each page with regex, verify target file exists
# Report orphaned pages (no incoming links)

## KHCC connection

This wiki could become a living KHCC oncology knowledge base maintained collaboratively by physicians and AI — every tumor board updates the relevant pages, every protocol revision shows up in the log, and any clinician can grep, edit, and trust the source of truth.